In [2]:
import tensorflow as tf

print(tf.__version__)

2.20.0


In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

import tensorflow as tf

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Dense,
    Dropout
)

In [4]:
path = "/content/drive/MyDrive/Customer-Churn-Prediction/data/processed_churn.csv"

df = pd.read_csv(path)

In [7]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,0,0,1,0,0,29.85,29.85,0,...,0,0,0,0,0,0,0,0,1,0
1,1,0,1,0,34,1,1,56.95,1889.50,0,...,0,0,0,0,0,1,0,0,0,1
2,1,0,1,0,2,1,0,53.85,108.15,1,...,0,0,0,0,0,0,0,0,0,1
3,1,0,1,0,45,0,1,42.30,1840.75,0,...,1,0,0,0,0,1,0,0,0,0
4,0,0,1,0,2,1,0,70.70,151.65,1,...,0,0,0,0,0,0,0,0,1,0


In [8]:
X = df.drop("Churn", axis=1)

y = df["Churn"]

In [9]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)

In [10]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [11]:
X_test_scaled

array([[-1.00374033, -0.43931886, -1.02831173, ...,  1.90515869,
        -0.71617747, -0.5394682 ],
       [-1.00374033, -0.43931886,  0.97246775, ..., -0.52489066,
        -0.71617747, -0.5394682 ],
       [-1.00374033, -0.43931886,  0.97246775, ..., -0.52489066,
        -0.71617747,  1.85367739],
       ...,
       [ 0.99627361, -0.43931886,  0.97246775, ..., -0.52489066,
        -0.71617747,  1.85367739],
       [ 0.99627361, -0.43931886,  0.97246775, ..., -0.52489066,
        -0.71617747,  1.85367739],
       [-1.00374033, -0.43931886,  0.97246775, ...,  1.90515869,
        -0.71617747, -0.5394682 ]])

In [14]:
ann = Sequential()
ann.add(Dense(
    units=64,
    activation="relu",
    input_dim=X_train.shape[1]
))
ann.add(Dropout(0.3))
ann.add(Dense(
    units=32,
    activation="relu"
))
ann.add(Dropout(0.2))
ann.add(Dense(
    units=1,
    activation="sigmoid"
))

In [15]:
ann.compile(

    optimizer="adam",

    loss="binary_crossentropy",

    metrics=["accuracy"]
)

In [16]:
history = ann.fit(

    X_train_scaled,
    y_train,

    validation_split=0.2,

    epochs=50,

    batch_size=32
)

Epoch 1/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.7576 - loss: 0.4937 - val_accuracy: 0.7876 - val_loss: 0.4214
Epoch 2/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7858 - loss: 0.4551 - val_accuracy: 0.7991 - val_loss: 0.4130
Epoch 3/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7942 - loss: 0.4392 - val_accuracy: 0.8036 - val_loss: 0.4094
Epoch 4/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8022 - loss: 0.4339 - val_accuracy: 0.7973 - val_loss: 0.4095
Epoch 5/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8013 - loss: 0.4229 - val_accuracy: 0.8018 - val_loss: 0.4085
Epoch 6/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8002 - loss: 0.4261 - val_accuracy: 0.8000 - val_loss: 0.4117
Epoch 7/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8053 - loss: 0.4195 - val_accuracy: 0.8044 - val_loss: 0.4120
Epoch 8/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8073 - loss: 0.4157 - val_accuracy: 0

In [17]:
ann_prob = ann.predict(X_test_scaled)

44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [18]:
ann_prob

array([[0.00651   ],
       [0.7689172 ],
       [0.00314063],
       ...,
       [0.01121215],
       [0.00861332],
       [0.00280582]], dtype=float32)

In [19]:
ann_pred = (ann_prob >= 0.5).astype(int)

In [20]:
ann_pred

array([[0],
       [1],
       [0],
       ...,
       [0],
       [0],
       [0]])

In [22]:
print(accuracy_score(y_test, ann_pred))
print(classification_report(y_test, ann_pred))
confusion_matrix(y_test, ann_pred)

0.7924662402274343
              precision    recall  f1-score   support

           0       0.85      0.88      0.86      1033
           1       0.62      0.56      0.59       374

    accuracy                           0.79      1407
   macro avg       0.73      0.72      0.72      1407
weighted avg       0.79      0.79      0.79      1407



array([[907, 126],
       [166, 208]])